# 03 - Painel Gerencial da Solução de Recomendação

Objetivo deste notebook: demonstrar, com indicadores e gráficos comparativos, a oportunidade de negócio para uma solução de recomendação de produtos.

Este notebook não treina modelo e não apresenta métricas de performance de modelo. O foco é mostrar comportamento atual, estratégia proposta e cenários financeiros conservadores.

## Os 3 pontos do painel

1. **Como estão os usuários hoje**: onde compram, o que compram, frequência, horário e consumo estimado.
2. **Como vamos resolver**: quais sinais serão usados para gerar recomendações.
3. **Projeções de consumo**: comparação entre comportamento atual e cenários conservadores de impacto financeiro.

## Configuração

A análise usa a tabela `training_data`, o catálogo original da Instacart e a base auxiliar de preços estimados. Como a Instacart não fornece preços no dataset original, os valores financeiros são aproximações baseadas em preços médios de mercado.

In [ ]:
from itertools import combinations
from pathlib import Path
import importlib
import os
import sys
import tempfile

import pandas as pd

# Mantém o cache do matplotlib fora do repositório.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(Path(tempfile.gettempdir()) / "fontconfig"))


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "instacart"
DATABASE_PATH = PROJECT_ROOT / "data" / "training_data.db"
PRICE_REFERENCE_PATH = PROJECT_ROOT / "data" / "reference" / "estimated_market_prices.csv"
ML_PREP_KIT_SRC = PROJECT_ROOT / "ml_prep_kit" / "src"

if str(ML_PREP_KIT_SRC) not in sys.path:
    sys.path.insert(0, str(ML_PREP_KIT_SRC))

# Recarrega o pacote local durante o desenvolvimento do notebook.
# Isso evita usar versões antigas das classes mantidas em memória pelo kernel.
import ml_prep_kit
import ml_prep_kit.visualization_reporter as visualization_reporter_module

importlib.reload(visualization_reporter_module)
importlib.reload(ml_prep_kit)

from ml_prep_kit import (
    CSVDataLoader,
    SQLiteDataFrameStore,
    VisualizationReporter,
    format_currency,
    format_percent,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
pd.set_option("display.float_format", "{:.4f}".format)

store = SQLiteDataFrameStore(DATABASE_PATH)
loader = CSVDataLoader(RAW_DATA_DIR)
reporter = VisualizationReporter()

TABLE_NAME = "training_data"
SAMPLE_USER_LIMIT = 3000
MODERATE_UPLIFT = 0.03
COOCCURRENCE_SAMPLE_ROWS = 1_000_000

source_columns = [
    "candidate_from_history",
    "candidate_from_cooccurrence",
    "candidate_from_favorite_category",
    "candidate_from_similar_users",
]

selected_columns = [
    "user_id",
    "product_id",
    "department",
    "aisle",
    "candidate_is_new_product_for_user",
    "candidate_was_previously_purchased",
    "user_total_orders",
    "user_total_items",
    "user_unique_products",
    "user_reorder_rate",
    "user_avg_days_between_orders",
    "user_avg_basket_size",
    "product_total_orders",
    "product_unique_users",
] + source_columns


In [ ]:
sample_filter = f"""
user_id IN (
    SELECT DISTINCT user_id
    FROM {TABLE_NAME}
    LIMIT {SAMPLE_USER_LIMIT}
)
"""

data = store.load_dataframe(
    table_name=TABLE_NAME,
    columns=selected_columns,
    where=sample_filter,
)

catalog_tables = loader.load(
    {
        "products": "products.csv",
        "aisles": "aisles.csv",
        "departments": "departments.csv",
    }
)

product_catalog = (
    catalog_tables["products"]
    .merge(catalog_tables["aisles"], on="aisle_id", how="left")
    .merge(catalog_tables["departments"], on="department_id", how="left")
)

data = data.merge(
    product_catalog[["product_id", "product_name"]],
    on="product_id",
    how="left",
)

try:
    price_reference = store.load_dataframe("estimated_market_prices")
except Exception:
    price_reference = pd.read_csv(PRICE_REFERENCE_PATH)

data.head().rename(
    columns={
        "user_id": "Usuário",
        "product_id": "Produto",
        "department": "Departamento",
        "aisle": "Corredor",
        "candidate_is_new_product_for_user": "Produto novo para o usuário",
        "candidate_was_previously_purchased": "Produto já comprado",
        "user_total_orders": "Total de pedidos do usuário",
        "user_total_items": "Total de itens do usuário",
        "user_unique_products": "Produtos únicos do usuário",
        "user_reorder_rate": "Taxa de recompra do usuário",
        "user_avg_days_between_orders": "Média de dias entre pedidos",
        "user_avg_basket_size": "Tamanho médio da cesta",
        "product_total_orders": "Total de pedidos do produto",
        "product_unique_users": "Usuários únicos do produto",
        "candidate_from_history": "Sinal por histórico",
        "candidate_from_cooccurrence": "Sinal por compra conjunta",
        "candidate_from_favorite_category": "Sinal por categoria favorita",
        "candidate_from_similar_users": "Sinal por usuários parecidos",
        "product_name": "Nome do produto",
    }
)


# 1. Como estão os usuários hoje

Esta seção usa a tabela `training_data`, enriquecida com o catálogo original da Instacart e com a base auxiliar `estimated_market_prices`. O objetivo é explicar o comportamento atual dos usuários antes de qualquer modelo: volume de pedidos, variedade de produtos, departamentos consumidos, horário médio e consumo financeiro estimado.


In [ ]:
item_prices = price_reference.loc[price_reference["unit"].ne("percent change")].copy()

price_by_aisle = (
    item_prices.groupby(["department", "aisle"], as_index=False)["estimated_price_usd"]
    .mean()
    .rename(columns={"estimated_price_usd": "aisle_estimated_price"})
)

price_by_department = (
    item_prices.groupby("department", as_index=False)["estimated_price_usd"]
    .mean()
    .rename(columns={"estimated_price_usd": "department_estimated_price"})
)

global_estimated_price = item_prices["estimated_price_usd"].mean()

data = data.merge(price_by_aisle, on=["department", "aisle"], how="left")
data = data.merge(price_by_department, on="department", how="left")
data["estimated_item_price"] = (
    data["aisle_estimated_price"]
    .fillna(data["department_estimated_price"])
    .fillna(global_estimated_price)
)

user_profile = (
    data.groupby("user_id", as_index=False)
    .agg(
        user_total_orders=("user_total_orders", "max"),
        user_total_items=("user_total_items", "max"),
        user_unique_products=("user_unique_products", "max"),
        user_reorder_rate=("user_reorder_rate", "max"),
        user_avg_days_between_orders=("user_avg_days_between_orders", "max"),
        user_avg_basket_size=("user_avg_basket_size", "max"),
        avg_estimated_item_price=("estimated_item_price", "mean"),
    )
)

user_profile["estimated_current_spend"] = (
    user_profile["user_total_items"] * user_profile["avg_estimated_item_price"]
)
user_profile["estimated_ticket_per_order"] = (
    user_profile["estimated_current_spend"] / user_profile["user_total_orders"]
)

baseline_spend = user_profile["estimated_current_spend"].sum()
baseline_ticket = user_profile["estimated_ticket_per_order"].mean()

reporter.create_kpi_table(
    [
        {
            "Indicador": "Usuários analisados",
            "Valor": f"{user_profile['user_id'].nunique():,}",
            "Observação": "Quantidade de usuários considerados na amostra do painel.",
        },
        {
            "Indicador": "Pedidos históricos",
            "Valor": f"{user_profile['user_total_orders'].sum():,.0f}",
            "Observação": "Total de pedidos já realizados pelos usuários analisados.",
        },
        {
            "Indicador": "Ticket médio estimado",
            "Valor": format_currency(baseline_ticket),
            "Observação": "Valor médio estimado por pedido, calculado com preços médios de mercado.",
        },
        {
            "Indicador": "Consumo atual estimado",
            "Valor": format_currency(baseline_spend),
            "Observação": "Valor financeiro aproximado do histórico de consumo da amostra.",
        },
    ]
)


### Consumo por departamento

**Base usada:** `training_data`, catálogo de produtos e `estimated_market_prices`.

**Transformação:** agrupamos os candidatos por `department`, somamos o `estimated_item_price` e calculamos um valor esperado aplicando o cenário moderado de incremento.

**Justificativa:** este gráfico mostra onde existe maior valor financeiro estimado hoje. A estratégia de recomendação deve priorizar departamentos com alto consumo e boa afinidade com o histórico do usuário. Se usuários que compram produtos do departamento B também costumam comprar produtos do departamento A, a recomendação pode antecipar essa relação e aumentar o consumo esperado no departamento A.


In [ ]:
department_values = (
    data.groupby("department", as_index=False)
    .agg(
        current_value=("estimated_item_price", "sum"),
        products=("product_id", "nunique"),
    )
    .sort_values("current_value", ascending=False)
    .head(10)
)
department_values["expected_value"] = department_values["current_value"] * (1 + MODERATE_UPLIFT)
department_values["incremental_value"] = department_values["expected_value"] - department_values["current_value"]

reporter.plot_comparison_bar(
    department_values,
    label_column="department",
    current_column="current_value",
    expected_column="expected_value",
    title="Consumo estimado por departamento: atual vs esperado",
    xlabel="Valor estimado em dólar",
)

department_values.rename(
    columns={
        "department": "Departamento",
        "current_value": "Valor atual estimado",
        "products": "Produtos distintos",
        "expected_value": "Valor esperado estimado",
        "incremental_value": "Valor incremental estimado",
    }
)


# 2. Como vamos resolver

Esta seção usa sinais já presentes na base preparada e relações observadas nos pedidos reais. Ainda não decidimos o modelo, então os gráficos não representam performance. Eles justificam a estratégia de geração de candidatos: histórico do usuário, produtos comprados juntos, categorias favoritas e comportamento de usuários parecidos.


### Sinais disponíveis para recomendação

**Base usada:** `training_data`, nas colunas `candidate_from_history`, `candidate_from_cooccurrence`, `candidate_from_favorite_category` e `candidate_from_similar_users`.

**Transformação:** calculamos a média de cada flag para medir a cobertura de cada fonte de candidato.

**Justificativa:** este gráfico mostra se temos fontes suficientes para recomendar. A solução não depende de uma regra única: se um usuário comprou B, podemos buscar A por coocorrência; se ele compra uma categoria com frequência, podemos sugerir produtos novos dessa categoria; se usuários parecidos compram A, A também pode ser candidato.


In [ ]:
source_labels = {
    "candidate_from_history": "Histórico do usuário",
    "candidate_from_cooccurrence": "Vendidos juntos",
    "candidate_from_favorite_category": "Categorias favoritas",
    "candidate_from_similar_users": "Usuários parecidos",
}

source_coverage = (
    data[source_columns]
    .mean()
    .rename("coverage")
    .reset_index()
    .rename(columns={"index": "source"})
)
source_coverage["source"] = source_coverage["source"].map(source_labels)

reporter.plot_ranking_bar(
    source_coverage,
    label_column="source",
    value_column="coverage",
    title="Sinais disponíveis para gerar recomendações",
    xlabel="Cobertura na base de candidatos",
)

source_coverage.rename(
    columns={
        "source": "Sinal de recomendação",
        "coverage": "Cobertura na base",
    }
)


### Associações de compra no mesmo pedido

**Base usada:** `order_products__prior.csv` e catálogo de produtos.

**Transformação:** lemos uma amostra dos itens de pedidos anteriores, associamos cada produto ao seu departamento e contamos pares únicos de departamentos que aparecem no mesmo `order_id`. Em seguida, mostramos uma matriz de intensidade: quanto mais escura a célula, mais vezes aquelas duas categorias aparecem juntas no mesmo pedido.

**Justificativa:** este gráfico é mais adequado para associação porque compara várias combinações ao mesmo tempo, sem repetir a direção `A -> B` e `B -> A`. Ele mostra, por exemplo, que `dairy eggs` e `produce` aparecem juntos em muitos pedidos. Essa evidência sustenta a estratégia de recomendar produtos complementares com base em afinidade de carrinho.


In [ ]:
# Usa uma amostra dos itens de pedidos anteriores para identificar categorias compradas juntas.
order_products_sample = pd.read_csv(
    RAW_DATA_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    nrows=COOCCURRENCE_SAMPLE_ROWS,
)

order_departments = order_products_sample.merge(
    product_catalog[["product_id", "department"]],
    on="product_id",
    how="left",
)

pair_counts = {}
for _, departments_in_order in order_departments.groupby("order_id")["department"]:
    basket_departments = sorted(set(departments_in_order.dropna()))
    if len(basket_departments) < 2:
        continue

    for left, right in combinations(basket_departments, 2):
        pair_counts[(left, right)] = pair_counts.get((left, right), 0) + 1

category_associations = pd.DataFrame(
    [
        {
            "source_department": left,
            "associated_department": right,
            "department_pair": f"{left} + {right}",
            "orders_together": count,
        }
        for (left, right), count in pair_counts.items()
        if count >= 500
    ]
)

top_category_associations = (
    category_associations.sort_values("orders_together", ascending=False)
    .head(12)
    .reset_index(drop=True)
)

reporter.plot_heatmap(
    top_category_associations,
    row_column="source_department",
    column_column="associated_department",
    value_column="orders_together",
    title="Matriz de categorias compradas juntas",
    colorbar_label="Pedidos juntos",
)

top_category_associations[
    [
        "department_pair",
        "orders_together",
    ]
].rename(
    columns={
        "department_pair": "Categorias compradas juntas",
        "orders_together": "Pedidos em comum",
    }
)


### Produtos com maior força histórica

**Base usada:** `training_data`, usando `product_total_orders`, `product_unique_users` e catálogo de produtos.

**Transformação:** selecionamos produtos únicos, ordenamos pelo volume histórico de pedidos e simulamos uma exposição esperada no cenário moderado.

**Justificativa:** produtos fortes no histórico ajudam a iniciar recomendações. Eles não devem ser recomendados sozinhos apenas por popularidade, mas funcionam como pontos de partida para encontrar produtos relacionados. Exemplo: clientes que compram B podem receber A quando A aparece nos mesmos pedidos, na mesma categoria de interesse ou no histórico de usuários parecidos.


In [ ]:
top_products = (
    data.sort_values("product_total_orders", ascending=False)
    .drop_duplicates("product_id")
    [["product_name", "department", "product_total_orders", "product_unique_users"]]
    .head(10)
)
top_products["expected_exposure"] = top_products["product_total_orders"] * (1 + MODERATE_UPLIFT)

reporter.plot_comparison_bar(
    top_products,
    label_column="product_name",
    current_column="product_total_orders",
    expected_column="expected_exposure",
    title="Produtos mais fortes no histórico: volume atual vs exposição esperada",
    xlabel="Volume histórico",
)

top_products.rename(
    columns={
        "product_name": "Produto",
        "department": "Departamento",
        "product_total_orders": "Total histórico de pedidos",
        "product_unique_users": "Usuários únicos",
        "expected_exposure": "Exposição esperada",
    }
)


# 3. Projeções de consumo

Esta seção usa o consumo estimado calculado a partir da `training_data` e da base `estimated_market_prices`. Como a base Instacart não possui preço, os valores financeiros são aproximações. As projeções não são resultado de modelo; são cenários para dimensionar o potencial financeiro da estratégia.


### Cenários conservadores de impacto incremental

**Base usada:** consumo atual estimado da amostra, calculado com `user_total_items` e `estimated_item_price`.

**Transformação:** aplicamos três percentuais pequenos sobre o consumo atual: 0,5%, 1,0% e 2,0%. Esses percentuais não representam resultado de modelo. Eles servem apenas para dimensionar quanto valor adicional poderia existir se parte das recomendações geradas por afinidade fosse aceita.

**Justificativa:** esta visão é mais conservadora do que uma linha do tempo. Ela não afirma que o consumo crescerá mês a mês. O gráfico responde uma pergunta mais simples e defensável: se a solução converter uma pequena parte das oportunidades observadas nos dados, qual seria o impacto financeiro estimado?


In [ ]:
scenario_plan = pd.DataFrame(
    [
        {"scenario": "Baixo impacto", "uplift_rate": 0.005},
        {"scenario": "Impacto moderado", "uplift_rate": 0.010},
        {"scenario": "Impacto alto", "uplift_rate": 0.020},
    ]
)
scenario_plan["current_value"] = baseline_spend
scenario_plan["incremental_value"] = baseline_spend * scenario_plan["uplift_rate"]
scenario_plan["expected_value"] = (
    scenario_plan["current_value"] + scenario_plan["incremental_value"]
)
scenario_plan["uplift_percent"] = scenario_plan["uplift_rate"].apply(format_percent)

reporter.plot_ranking_bar(
    scenario_plan,
    label_column="scenario",
    value_column="incremental_value",
    title="Cenários conservadores: valor incremental estimado",
    xlabel="Valor incremental estimado em dólar",
)

scenario_plan[
    [
        "scenario",
        "uplift_percent",
        "current_value",
        "incremental_value",
        "expected_value",
    ]
].rename(
    columns={
        "scenario": "Cenário",
        "uplift_percent": "Incremento considerado",
        "current_value": "Valor atual estimado",
        "incremental_value": "Valor incremental estimado",
        "expected_value": "Valor esperado estimado",
    }
)


## Conclusão do projeto

O notebook mostra oportunidade e direção, não resultado de modelo. A análise está apoiada em comportamento real, preços estimados com fonte documentada e cenários conservadores para dimensionar o potencial financeiro.

O próximo passo é decidir a abordagem de recomendação, definir a separação temporal de validação e só então medir performance com métricas adequadas de ranking.